In [2]:
import requests
import pandas as pd
import re
import time

stations_dms = {
    "Barisal":     ("22°43'", "90°22'"),
    "Bhola":       ("22°41'", "90°39'"),
    "Bogra":       ("24°51'", "89°22'"),
    "Chandpur":    ("23°14'", "90°42'"),
    "Chittagong":  ("22°13'", "91°48'"),   
    "chuadanga":   ("23°39'", "88°49'"),   
    "Comilla":     ("23°26'", "91°11'"),
    "CoxsBazar":   ("21°27'", "91°58'"),   
    "Dhaka":       ("23°46'", "90°23'"),   
    "Dinajpur":    ("25°39'", "88°41'"),
    "Faridpur":    ("23°36'", "89°51'"),
    "Feni":        ("23°02'", "91°25'"),
    "Hatiya":      ("22°27'", "91°06'"),
    "Ishurdi":     ("24°09'", "89°02'"),   
    "Jessore":     ("23°12'", "89°20'"),
    "Khepupara":   ("21°59'", "90°41'"),
    "Khulna":      ("22°47'", "89°34'"),
    "Kutubdia":    ("21°49'", "91°51'"),
    "Madaripur":   ("23°10'", "90°11'"),
    "M.court":     ("22°52'", "91°06'"),   
    "Mongla":      ("22°28'", "89°36'"),
    "Mymensingh":  ("24°44'", "90°25'"),
    "Patuakhali":  ("22°20'", "90°20'"),
    "Rajshahi":    ("24°22'", "88°42'"),
    "Rangpur":     ("25°44'", "89°16'"),
    "Rangamati":   ("22°22'", "92°09'"),
    "Sandwip":     ("22°29'", "91°26'"),
    "Satkhira":    ("22°43'", "89°05'"),
    "Sitakunda":   ("22°38'", "91°42'"),
    "Srimangal":   ("24°18'", "91°44'"),
    "sydpur":      ("25°45'", "88°55'"),   
    "Sylhet":      ("24°54'", "91°53'"),
    "Tangail":     ("24°15'", "89°56'"),
    "Teknaf":      ("20°52'", "92°18'"),
}

def dms_to_decimal(dms_str):
    match = re.match(r"(\d+)[°]\s*(\d+)?", dms_str)
    deg = int(match.group(1))
    minutes = int(match.group(2)) if match.group(2) else 0
    return round(deg + minutes / 60, 4)

stations = {
    name: (dms_to_decimal(lat), dms_to_decimal(lon))
    for name, (lat, lon) in stations_dms.items()
}

# Finalized parameters
PARAMS = "T2M,RH2M,PS,WS2M,CLOUD_AMT,ALLSKY_SFC_SW_DWN,T2MDEW"
START = "19840101"
END = "20161231"
URL = "https://power.larc.nasa.gov/api/temporal/daily/point"

def get_nasa_station(name, lat, lon, retries=3):
    params = {
        "parameters": PARAMS,
        "community": "AG",
        "longitude": lon,
        "latitude": lat,
        "start": START,
        "end": END,
        "format": "JSON",
    }
    for attempt in range(retries):
        try:
            r = requests.get(URL, params=params, timeout=60)
            r.raise_for_status()
            data = r.json()
            df = pd.DataFrame(data["properties"]["parameter"])
            df.index = pd.to_datetime(df.index, format="%Y%m%d")
            df = df.reset_index().rename(columns={"index": "date"})
            df["Station"] = name
            return df
        except Exception as e:
            print(f"  attempt {attempt+1} failed for {name}: {e}")
            time.sleep(5)
    print(f"  FAILED permanently: {name}")
    return None

all_data = []
for name, (lat, lon) in stations.items():
    print(f"Fetching {name} ({lat}, {lon})...")
    df = get_nasa_station(name, lat, lon)
    if df is not None:
        all_data.append(df)
    time.sleep(1)

full_df = pd.concat(all_data, ignore_index=True)
full_df = full_df.replace(-999, pd.NA)
full_df.to_csv(r"Data\Raw\nasa_power_1984_2016.csv", index=False)

print(f"\nDone. Total rows: {len(full_df)}")

Stations loaded: 34 (expect 34 - currently missing Ambagan_ctg)
Barisal: lat=22.7167, lon=90.3667
Bhola: lat=22.6833, lon=90.65
Bogra: lat=24.85, lon=89.3667
Chandpur: lat=23.2333, lon=90.7
Chittagong: lat=22.2167, lon=91.8
chuadanga: lat=23.65, lon=88.8167
Comilla: lat=23.4333, lon=91.1833
CoxsBazar: lat=21.45, lon=91.9667
Dhaka: lat=23.7667, lon=90.3833
Dinajpur: lat=25.65, lon=88.6833
Faridpur: lat=23.6, lon=89.85
Feni: lat=23.0333, lon=91.4167
Hatiya: lat=22.45, lon=91.1
Ishurdi: lat=24.15, lon=89.0333
Jessore: lat=23.2, lon=89.3333
Khepupara: lat=21.9833, lon=90.6833
Khulna: lat=22.7833, lon=89.5667
Kutubdia: lat=21.8167, lon=91.85
Madaripur: lat=23.1667, lon=90.1833
M.court: lat=22.8667, lon=91.1
Mongla: lat=22.4667, lon=89.6
Mymensingh: lat=24.7333, lon=90.4167
Patuakhali: lat=22.3333, lon=90.3333
Rajshahi: lat=24.3667, lon=88.7
Rangpur: lat=25.7333, lon=89.2667
Rangamati: lat=22.3667, lon=92.15
Sandwip: lat=22.4833, lon=91.4333
Satkhira: lat=22.7167, lon=89.0833
Sitakunda: lat=